# Pipeline de preprocesamiento conductual

`participants.tsv` -> limpieza (parsear `n/a`, tipado) -> filtrado (quedarse
solo con sujetos con dato pareado NS+SD, n documentado por variable) ->
derivación (`delta = SD - NS` por marcador) -> `behavioral_deltas.csv`.

Correr desde el entorno de Python (kernel) de la raíz del repo, de arriba
hacia abajo.

In [1]:
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve().parents[2]  # deliveries/week06/code -> raíz del repo
PARTICIPANTS_TSV = REPO_ROOT / "deliveries" / "week04" / "data" / "participants.tsv"
OUT_DIR = REPO_ROOT / "deliveries" / "week06" / "data" / "processed"
REPO_ROOT

WindowsPath('C:/Users/jimen/Downloads/data-visualization-project')

In [2]:
# Marcadores con par Sueño Normal (NS) / Privación de Sueño (SD), según el
# data dictionary (deliveries/week04/data_dictionary.csv). Un SD-NS mayor
# significa que el marcador empeoró (más lento/menos alerta/afecto más
# negativo) tras la privación de sueño, excepto PANAS_P y PANAS_N que
# siguen su propia polaridad (ver notas en DataAnalysis.md).
PAIRED_MARKERS = {
    "PVT_lapses": ("PVT_item1_NS", "PVT_item1_SD"),
    "PVT_medianRT": ("PVT_item2_NS", "PVT_item2_SD"),
    "PVT_sdRT": ("PVT_item3_NS", "PVT_item3_SD"),
    "PANAS_P": ("PANAS_P_NS", "PANAS_P_SD"),
    "PANAS_N": ("PANAS_N_NS", "PANAS_N_SD"),
    "ATQ": ("ATQ_NS", "ATQ_SD"),
    "SAI": ("SAI_NS", "SAI_SD"),
    "SSS": ("SSS_NS", "SSS_SD"),
    "KSS": ("KSS_NS", "KSS_SD"),
}

# Moderadores a nivel de participante (Q7): sexo, edad, calidad de sueño
# habitual, orden de sesión.
MODERATOR_COLS = ["Gender", "Age", "SessionOrder", "PSQI_GlobalScore"]

In [3]:
df = pd.read_csv(PARTICIPANTS_TSV, sep="\t", na_values=["n/a", "N/A", ""])
df.head()

,participant_id,Gender,Age,SessionOrder,EEG_SamplingTime_Open_NS,EEG_SamplingTime_Closed_NS,EEG_SamplingTime_Open_SD,EEG_SamplingTime_Closed_SD,PVT_SamplingTime_NS,PVT_SamplingTime_SD,...,EQ,Buss_Perry,PSQI_GlobalScore,PSQI_item1,PSQI_item2,PSQI_item3,PSQI_item4,PSQI_item5,PSQI_item6,PSQI_item7
0,sub-01,M,22,NS->SD,8:57:52,9:04:40,8:18:28,8:25:34,9:27:23,9:23:08,...,NaN,NaN,4.0,1.0,2.0,0.0,0.0,1.0,0.0,0.0
1,sub-02,M,21,NS->SD,9:47:46,9:53:24,8:54:10,8:59:50,10:16:59,10:00:47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sub-03,F,19,NS->SD,8:49:46,8:55:38,9:58:36,10:04:08,8:45:04,9:34:13,...,NaN,NaN,7.0,2.0,1.0,0.0,1.0,2.0,0.0,1.0
3,sub-04,M,22,NS->SD,9:53:56,9:59:34,9:07:02,9:12:26,10:05:35,10:06:28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sub-05,F,18,NS->SD,20:37:12,20:47:34,9:09:00,9:14:34,21:01:57,10:07:33,...,NaN,NaN,3.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0


In [4]:
def derive_deltas(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df[["participant_id", *MODERATOR_COLS]].copy()
    n_rows = []

    for label, (ns_col, sd_col) in PAIRED_MARKERS.items():
        paired_mask = df[ns_col].notna() & df[sd_col].notna()
        delta_col = f"delta_{label}"
        out[delta_col] = pd.NA
        out.loc[paired_mask, delta_col] = df.loc[paired_mask, sd_col] - df.loc[paired_mask, ns_col]
        n_rows.append(
            {
                "marker": label,
                "n_ns_only": int(df[ns_col].notna().sum()),
                "n_sd_only": int(df[sd_col].notna().sum()),
                "n_paired": int(paired_mask.sum()),
            }
        )

    paired_n_summary = pd.DataFrame(n_rows).sort_values("n_paired", ascending=False)
    return out, paired_n_summary


deltas, paired_n_summary = derive_deltas(df)
paired_n_summary

,marker,n_ns_only,n_sd_only,n_paired
4,PANAS_N,71,68,68
3,PANAS_P,71,68,68
7,SSS,38,37,35
8,KSS,33,33,33
6,SAI,31,31,31
0,PVT_lapses,31,37,30
2,PVT_sdRT,31,37,30
1,PVT_medianRT,31,37,30
5,ATQ,27,25,25


In [5]:
deltas.head()

,participant_id,Gender,Age,SessionOrder,PSQI_GlobalScore,delta_PVT_lapses,delta_PVT_medianRT,delta_PVT_sdRT,delta_PANAS_P,delta_PANAS_N,delta_ATQ,delta_SAI,delta_SSS,delta_KSS
0,sub-01,M,22,NS->SD,4.0,<NA>,<NA>,<NA>,-5.0,1.0,<NA>,<NA>,2.0,<NA>
1,sub-02,M,21,NS->SD,NaN,-4.0,28.5,-22.21,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,sub-03,F,19,NS->SD,7.0,-3.0,44.0,21.07,-6.0,7.0,<NA>,<NA>,<NA>,<NA>
3,sub-04,M,22,NS->SD,NaN,<NA>,<NA>,<NA>,-2.0,-1.0,<NA>,<NA>,1.0,<NA>
4,sub-05,F,18,NS->SD,3.0,6.0,68.5,49.54,-6.0,-1.0,<NA>,<NA>,1.0,<NA>


In [6]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
deltas_path = OUT_DIR / "behavioral_deltas.csv"
summary_path = OUT_DIR / "paired_n_summary.csv"
deltas.to_csv(deltas_path, index=False)
paired_n_summary.to_csv(summary_path, index=False)
print(f"Se escribió {deltas_path} ({len(deltas)} participantes)")
print(f"Se escribió {summary_path}")

Se escribió C:\Users\jimen\Downloads\data-visualization-project\deliveries\week06\data\processed\behavioral_deltas.csv (71 participantes)
Se escribió C:\Users\jimen\Downloads\data-visualization-project\deliveries\week06\data\processed\paired_n_summary.csv
